# 1. What you'll learn

        - read thresholds and leaves from the plotted tree
- measure depth's train/test trade-off
- show root-split instability under bootstrap resampling

        **The one question this notebook answers:** How do greedy yes/no rules predict, and why can a small sample change rewrite the tree?

# 2. Setup

This lesson keeps decision trees small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier,plot_tree
from sklearn.metrics import balanced_accuracy_score,ConfusionMatrixDisplay

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The Wisconsin breast-cancer dataset has 569 masses and 30 continuous measurements. We predict malignant versus benign.

**Small example:** For radii [8,10,12,16,18] labelled [B,B,B,M,M], radius≤14 creates pure children and drops Gini from 0.48 to 0.

In [ ]:
b=load_breast_cancer(as_frame=True);X=b.data;y=(b.target==0).astype(int);print("Shape:",X.shape,"malignant share:",round(y.mean(),3));print(X.iloc[:3,:5].round(2))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));y.value_counts().plot.bar(ax=axes[0],color="#176b66");axes[0].set_title("Class counts")
sns.histplot(x=X["worst radius"],hue=y,element="step",stat="density",common_norm=False,ax=axes[1]);axes[1].set_title("Useful threshold, imperfect overlap")
sns.heatmap(X.iloc[:,:10].corr(),cmap="vlag",center=0,xticklabels=False,yticklabels=False,ax=axes[2]);axes[2].set_title("Correlated measurements");plt.tight_layout();plt.show()

**Takeaway:** Malignancy is the minority, so balanced accuracy matters.

**Takeaway:** One radius threshold separates much of the population but not all.

**Takeaway:** Correlated measurements offer several nearly equivalent greedy splits.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED);depths=[1,2,3,5,None];models={d:DecisionTreeClassifier(max_depth=d,min_samples_leaf=3,random_state=SEED).fit(X_train,y_train) for d in depths};print("Train/test:",X_train.shape,X_test.shape)

# 6. Train

        Training turns the assumptions behind decision trees into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Put every training row at the root.
2. Try all feature thresholds and measure impurity reduction.
3. Take the locally best split.
4. Repeat until depth, leaf size, or purity stops growth.

In [ ]:
for d,m in models.items(): print(f"depth={str(d):4s} leaves={m.get_n_leaves():3d} train={m.score(X_train,y_train):.3f} test-balanced={balanced_accuracy_score(y_test,m.predict(X_test)):.3f}")
model=models[3];print("Root:",X.columns[model.tree_.feature[0]],"<=",round(model.tree_.threshold[0],3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
scores=[balanced_accuracy_score(y_test,models[d].predict(X_test)) for d in depths];fig,axes=plt.subplots(1,3,figsize=(16,4.5));axes[0].plot(range(5),scores,marker="o");axes[0].axhline(.5,ls="--",color="black");axes[0].set_xticks(range(5),[str(d) for d in depths]);axes[0].set_title("Test balanced accuracy")
plot_tree(model,max_depth=2,feature_names=X.columns.tolist(),class_names=["benign","malignant"],filled=True,fontsize=6,ax=axes[1]);axes[1].set_title("Actual first levels")
ConfusionMatrixDisplay.from_predictions(y_test,model.predict(X_test),ax=axes[2],colorbar=False);axes[2].set_title("Held-out confusion");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print("Baseline balanced accuracy=0.500")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Greedy splits are unstable when correlated predictors offer nearly equal impurity gains. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
rng=np.random.default_rng(SEED);roots=[];scores=[]
for s in range(60):
 idx=rng.choice(len(X_train),len(X_train),replace=True);m=DecisionTreeClassifier(max_depth=3,min_samples_leaf=3,random_state=s).fit(X_train.iloc[idx],y_train.iloc[idx]);roots.append(X.columns[m.tree_.feature[0]]);scores.append(balanced_accuracy_score(y_test,m.predict(X_test)))
print("Root features:",pd.Series(roots).value_counts().head().to_dict());print(f"Score range={min(scores):.3f}–{max(scores):.3f}")

**Use this when…** nonlinear interactions matter and a compact inspected rule set is valuable.

        **Don't use this when…** predictions must be smooth and stable or extrapolate outside training support.

        ## Try this

        1. Sweep min_samples_leaf and count leaves.
2. Remove the root feature and find its substitute.
3. Choose pruning alpha on validation only.